In [1]:
import matplotlib.pyplot as plt
from statsmodels.base.model import GenericLikelihoodModel
import cv2
import scienceplots
import tifffile as tiff

from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia
from boulder_statistics.analysis.external_data_encyclopedia import ExternalDataEncyclopedia
plt.style.use('science')
plt.rcParams["figure.figsize"] = (3.5, 3.5 * ((5**0.5 - 1) / 2)) # 3.5
plt.rcParams["figure.dpi"] = 600
%matplotlib inline

import polars as pl
from polars import Expr, LazyFrame, DataFrame
import numpy as np
from pathlib import Path
from typing import Any
import tifffile
from typing import Dict
from typing import Callable


from boulder_statistics.analysis.data_product_encyclopedia import DataProductEncyclopedia

dp = DataProductEncyclopedia(
    data_products_path=Path(r"C:\Users\Joshu\OneDrive - Nexus365\AO33\Boulder_database\AO33\.data_products"))

ed = ExternalDataEncyclopedia(
    external_data_path=Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data")
)

In [2]:
exports_folder = Path("dp_export_tests")
areas_export_path: Path = exports_folder / "00_add_areas"
clean_db_export_path: Path = exports_folder / "01_cleanup_db"
spectral_export_path: Path = exports_folder / "02_add_spectral"

In [5]:
import os
from pathlib import Path
from typing import List

from polars.dataframe.frame import DataFrame

from boulder_statistics.refinement_plus.facet_parser import FacetParser
from boulder_statistics.refinement_plus.bulk_parse_data_tir_maps import DataTirMaps
from boulder_statistics.refinement_plus.qcube_chunk import QCubeChunk
from boulder_statistics.refinement_plus.refinement_chunking import ChunkingTools

mesh_folder = Path(r"C:\Users\Joshu\Documents\AO33_backup\Boulder_database\AO33\external_data\bennu_models")
measurement_types: list[str] = ["band depth 350", "band depth 440", "slope 1000", "ratio 1000",
                                "sigma band depth 350", "sigma band depth 440", "sigma slope 1000", "sigma ratio 1000"]

for mesh_file in os.listdir(mesh_folder):
    mesh_file_path: Path = mesh_folder / mesh_file

    data_tir_maps = DataTirMaps.bulk_parse(ed).filter(
        pl.col("facet_shape_model_name") == mesh_file)

    points, tris = FacetParser.load_mesh(mesh_file_path)

    # assert tris.height == data_tir_maps.height

    tris_collected_with_facets: DataFrame = FacetParser.associate_mesh_tris_with_facet_num(
        data_tir_maps,
        tris
    )

    tris_to_draw: DataFrame = tris_collected_with_facets.join(
        data_tir_maps,
        on = "facet_num"
    ).with_row_index("tri_id").sort("tri_id") # Therefore for each tri-id there is the facet's data on a row

    measurement_type_triangle_values_dict: Dict[str, np.ndarray] = {
        measurement_type : (tris_to_draw.get_column(measurement_type).to_numpy())
        for measurement_type in measurement_types
    }

    def handle_chunk(chunk : QCubeChunk) -> List[np.ndarray]:
        triangle_index_image = FacetParser.rasterize_facets(
            points,
            tris_to_draw,
            chunk
        )

        measurement_types_results : List[np.ndarray] = []

        for measurement_type in measurement_types:
            output = np.full(triangle_index_image.shape, np.nan)
            mask = ~np.isnan(triangle_index_image)
            
            output[mask] = measurement_type_triangle_values_dict[measurement_type][
                triangle_index_image[mask].astype(np.int32)
            ]

            measurement_types_results.append(output)
        
        return measurement_types_results

    ChunkingTools.bulk_append_by_chunks(
        pl.scan_parquet(clean_db_export_path),
        spectral_export_path / Path(mesh_file).stem,
        measurement_types,
        handle_chunk,
        chunks = QCubeChunk.generate(depth=2)
    )

Processing chunks: 100%|██████████| 96/96 [04:05<00:00,  2.56s/it]
